In [1]:
import os
import sys
import json

In [ ]:
# Adding the bin directory inside my conda environment to PATH variable so that python 3.8 and etllib commands can be found and accessed
# User should do this on their end as well if necessary
os.environ["PATH"] = "/Users/kirthichillakanti/miniconda3/envs/dsci550_hw_1/bin:" + os.environ["PATH"]

In [ ]:
# Checking if previous step worked
!echo $PATH

In [ ]:
# Checking python verison
!python --version

In [ ]:
# Getting the path to scripts directory
# Creating a path for the output json file when we do tsvtojson
# Storing path of tsv file in tsv_data_file_path variable
scripts_dir =os.getcwd()
json_file_path = os.path.join(scripts_dir, "..", "output_json_file.json")
tsv_data_file_path = os.path.join(scripts_dir, "..", "data", "haunted_places.tsv")

In [ ]:
!tsvtojson -t {tsv_data_file_path} -j {json_file_path} -c ../conf/colheaders.conf -o haunted_places -s 0.8 -v -e ../conf/encoding.conf

In [ ]:
# creating a path to a directory called all_json_files to put all the individual json files
parent_of_scripts = os.path.join(scripts_dir, "..")
folder_path = os.path.join(parent_of_scripts, "all_json_files")
os.makedirs(folder_path, exist_ok = True)

In [ ]:
def split_aggregated_json(aggregated_json_path, output_base_dir, files_per_folder=100):
    """
    Reads an aggregated JSON file (with an outer wrapper) and writes each row as its own JSON file.
    Organizes the files into subfolders, each containing `files_per_folder` files.

    Args:
        aggregated_json_path (str): Path to the aggregated JSON file.
        output_base_dir (str): Directory where individual JSON files and subfolders will be created.
        files_per_folder (int): Number of JSON files per subfolder.
    """
    # Ensure the output base directory exists.
    if not os.path.exists(output_base_dir):
        os.makedirs(output_base_dir)

    # Load the aggregated JSON file.
    with open(aggregated_json_path, 'r', encoding='utf-8') as infile:
        data = json.load(infile)

    # If your aggregated JSON file is wrapped in an outer object (e.g., {"haunted_places": [ ... ]})
    # then get the list of rows. Otherwise, assume data is the list.
    if isinstance(data, dict):
        # Change "haunted_places" to whatever key your aggregated JSON uses.
        rows = data.get("haunted_places", [])
    else:
        rows = data

    # Loop through each row and write each one to its own file.
    for i, row in enumerate(rows):
        # Determine subfolder index (starting at 1).
        folder_index = (i // files_per_folder) + 1
        subfolder_name = f"dir_{folder_index:03d}"
        subfolder_path = os.path.join(output_base_dir, subfolder_name)

        # Create the subfolder if it doesn't exist.
        if not os.path.exists(subfolder_path):
            os.makedirs(subfolder_path)

        # Create a filename for the JSON file (e.g., row_000001.json).
        json_file_name = f"row_{i:06d}.json"
        json_file_path = os.path.join(subfolder_path, json_file_name)

        # Write the row data as a JSON file.
        with open(json_file_path, 'w', encoding='utf-8') as outfile:
            json.dump(row, outfile, indent=2)

        print(f"Created {json_file_path}")

    print(f"Processed {len(rows)} rows into individual JSON files in '{output_base_dir}'.")


# Example usage:
aggregated_json_path = json_file_path  # Path to your aggregated JSON file.
output_base_dir = folder_path          # Change as needed.
split_aggregated_json(aggregated_json_path, output_base_dir, files_per_folder=100)

In [ ]:
## RUN THESE IN TERMINAL

#pkill -f tika-server.jar
#java -jar ~/tika-server.jar
os.environ["TIKA_SERVER_JAR"] = "file:////Users/kirthichillakanti/tika-server-standard-3.1.0.jar"

In [ ]:
# Storing the path to the cosine_similarity.py file in a variable
cosine_similarity_py_path = os.path.join(scripts_dir, "..", "tika-img-similarity", "tikasimilarity", "distance", "cosine_similarity.py")

# Storing the path to the jaccard_similarity.py file in a variable
jaccard_similarity_py_path = os.path.join(scripts_dir, "..", "tika-img-similarity", "tikasimilarity", "distance", "jaccard_similarity.py")

# Storing the path to the edit_distance.py file in a variable
edit_distance_py_path = os.path.join(scripts_dir, "..", "tika-img-similarity", "tikasimilarity", "distance", "edit-value-similarity.py")


# Added execution permission to cosine_similarity.py file so that it can be executed
subprocess.run(['chmod', '+x', cosine_similarity_py_path]) 
# Added execution permission to jaccard_similarity.py file so that it can be executed
subprocess.run(['chmod', '+x', jaccard_similarity_py_path])
# Added execution permission to edit_distance.py file so that it can be executed
subprocess.run(['chmod', '+x', edit_distance_py_path])

In [ ]:
import subprocess
# The following functions are for cosine similarity, jaccard similarity, and edit_distance
# These functions take an integer as input.
# This integer input represents the directory number which contains a specific subset of data
# So we will run these similarity functions on a specific subset of data at a time

def cosine_similarity(i:int):
    # d_name stores the name of the directory that has the subset of data user is focused on
    d_name = f"dir_{i:03d}"
    
    # Creates a folder for the certain subset of data that we call this function on. 
    # This folder will contain the cosine_similarity matrix
    results_folder_path = os.path.join(parent_of_scripts, f"dir_{i}")
    os.makedirs(results_folder_path, exist_ok = True)
    
    # Creating a path for the output csv file
    output_csv_path = os.path.join(results_folder_path, "cosine_similarity_matrix.csv")
    
    # Executed the command to run the cosine_similarity.py on the subset of data
    subprocess.run(['python', cosine_similarity_py_path, '--inputDir', f"{folder_path}/{d_name}", '--outCSV', output_csv_path])
    
    # Returns the results_folder_path for use later on, when we want to put the clustering json/html files in the same folder.  
    return results_folder_path 

def jaccard_similarity(i:int):
    d_name = f"dir_{i:03d}"
    
    # Creates a folder for the certain subset of data that we call this function on. 
    # This folder will contain the jaccard similarity matrix
    results_folder_path = os.path.join(parent_of_scripts, f"dir_{i}")
    os.makedirs(results_folder_path, exist_ok = True)
    
    # Creating a path for the output csv file
    output_csv_path = os.path.join(results_folder_path, "jaccard_similarity_matrix.csv")
    
    # Executed the command to run the jaccard_similarity.py on the subset of data
    subprocess.run(['python', jaccard_similarity_py_path, '--inputDir', f"{folder_path}/{d_name}", '--outCSV', output_csv_path])

    # Returns the results_folder_path for use later on, when we want to put the clustering json/html files in the same folder. 
    return results_folder_path

def edit_distance(i:int):
    d_name = f"dir_{i:03d}"
    # Creates a folder for the certain subset of data that we call this function on. 
    # This folder will contain the edit distance similarity matrix
    results_folder_path = os.path.join(parent_of_scripts, f"dir_{i}")
    os.makedirs(results_folder_path, exist_ok = True)
    
    # Creating a path for the output csv file
    output_csv_path = os.path.join(results_folder_path, "edit_distance_matrix.csv")
    
    # Executed the command to run the edit_distance.py on the subset of data
    subprocess.run(['python', edit_distance_py_path, '--inputDir', f"{folder_path}/{d_name}", '--outCSV', output_csv_path])

    # Returns the results_folder_path for use later on, when we want to put the clustering json/html files in the same folder. 
    return results_folder_path

In [ ]:
results_folder_path = edit_distance(1)

In [ ]:
results_folder_path = cosine_similarity(1)

In [ ]:
results_folder_path = jaccard_similarity(1)

# Clustering on dir_001

In [ ]:
# Run edit-cosine-circle-packing.py 
# Puts the resulting output file into the same folder that contains the similarity matrix for this particular subset of data
edit_cosine_circle_packing_path = os.path.join(scripts_dir, "..", "tika-img-similarity", "tikasimilarity", "cluster", "edit-cosine-circle-packing.py")
!chmod +x {edit_cosine_circle_packing_path}
!cd {results_folder_path} && {edit_cosine_circle_packing_path} --inputCSV edit_distance_matrix.csv --cluster 0

In [ ]:
# Run edit-cosine-cluster.py
# Puts the resulting output file into the same folder that contains the similarity matrix for this particular subset of data
edit_cosine_cluster_path = os.path.join(scripts_dir, "..", "tika-img-similarity", "tikasimilarity", "cluster", "edit-cosine-cluster.py")
!chmod +x {edit_cosine_cluster_path}
!cd {results_folder_path} && {edit_cosine_cluster_path} --inputCSV edit_distance_matrix.csv --cluster 2

In [ ]:
# Run generateLevelCluster.py
# Puts the resulting output file into the same folder that contains the similarity matrix for this particular subset of data
generateLevelCluster_path = os.path.join(scripts_dir, "..", "tika-img-similarity", "tikasimilarity", "cluster", "generateLevelCluster.py")
!chmod +x {generateLevelCluster_path}
!cd {results_folder_path} && {generateLevelCluster_path}

In [ ]:
# Copy the files from etllib/html to cluster_result_path
import shutil
etllib_html_path = os.path.expanduser('~/etllib/html')
files = os.listdir(etllib_html_path)

for file in files:
    source = os.path.join(etllib_html_path, file)
    destination = os.path.join(results_folder_path, file)
    shutil.copy(source, destination)

In [ ]:
# Start a simple HTTP server to serve the files
!cd {results_folder_path} && python -m http.server 8083